In [12]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON

In [125]:
df = pd.read_csv("SQ_results_w_langgraph.csv")

In [126]:
df.head()

,SAE Question,AAVE Question,SPARQL,Results,Attempts
0,Who was the musician born in Detroit?,who a musician born in Detroit,SELECT ?item ?itemLabel WHERE {\n ?item wdt:P...,"{'head': {'vars': ['item', 'itemLabel']}, 'res...",1
1,"In what language was the movie ""Mera Shikar"" f...",what’s da language dat mera shikar was filmed in?,SELECT DISTINCT ?language ?languageLabel WHERE...,"{'head': {'vars': ['language', 'languageLabel'...",1
2,What is the name of a battle that happened in ...,Wat's da name of a battle dat happened in Chic...,SELECT ?item ?itemLabel WHERE {\n ?item wdt:P...,"{'head': {'vars': ['item', 'itemLabel']}, 'res...",1
3,Which player plays the position of midfielder?,Wha player play da position midfielder?,SELECT DISTINCT ?player ?playerLabel WHERE {\n...,"{'head': {'vars': ['player', 'playerLabel']}, ...",1
4,What position did Mike Twellman play?,wat position Mike Twellman play?,SELECT ?position ?positionLabel WHERE {\n ?pl...,"{'head': {'vars': ['position', 'positionLabel'...",1


In [123]:
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)


def execute_query(query):
    sparql.setQuery(query)
    results = sparql.query().convert()
    return results






In [127]:
names = []
queries = df['SPARQL'].tolist()
for i in range(len(queries)):
    query = str(queries[i])
    
    try:
        eq = execute_query(query)
        
        if not eq['results']['bindings']:
            names.append(None)
            print(f'Query {i+1} returned no results')
        else:
            keys = eq['results']['bindings'][0].keys()
            
            # First try to find keys ending with 'LabelLabel' 
            label_key = next((k for k in keys if k.endswith('LabelLabel')), None)
            
            # If not found, fall back to any key ending with 'Label'
            if label_key is None:
                label_key = next(k for k in keys if k.endswith('Label'))
            
            name = eq['results']['bindings'][0][label_key]['value']
            names.append(name)
            print(f'Query {i+1} processed: {name}')
            
    except Exception as e:
        names.append(None)
        print(f'Query {i+1} skipped - Malformed query')

Query 1 processed: Major Holley
Query 2 processed: Hindi
Query 3 processed: Battle of Fort Dearborn
Query 4 processed: Michael Johnson
Query 5 processed: defender
Query 6 returned no results
Query 7 processed: United States
Query 8 processed: English
Query 9 processed: James Henry
Query 10 returned no results
Query 11 returned no results
Query 12 skipped - Malformed query
Query 13 processed: Histoire de Melody Nelson
Query 14 returned no results
Query 15 processed: Endless Love
Query 16 returned no results
Query 17 processed: Arnold Schoenberg
Query 18 processed: basketball
Query 19 processed: Berlin
Query 20 processed: Western European Time
Query 21 processed: United States
Query 22 processed: female
Query 23 processed: Francis of Assisi
Query 24 processed: adventure video game
Query 25 returned no results
Query 26 processed: Q114949816
Query 27 processed: Canada
Query 28 processed: Sivas
Query 29 processed: male
Query 30 processed: Theodore Olson
Query 31 processed: Night Ranger
Quer

In [103]:
def calculate_f1(pred: List[Any], gold: List[Any]) -> float:
    if not pred or not gold:
        return float(pred == gold)

    pred = [str(x) for x in pred]
    gold = [str(x) for x in gold]

    common = collections.Counter(pred) & collections.Counter(gold)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred)
    recall = num_same / len(gold)
    return (2 * precision * recall) / (precision + recall)
